# Classification of b-quark jets in the Aleph simulated data

The following is an introduction to using Machine Learning (ML) - in particular Boosted Decision Trees (BDT) - for trying to determine, if an entry in a data file is of one type (signal, ill, guilty, etc.) or another (background, healthy, innocent, etc.).

You may choose between two data samples:
1. A particle physics dataset containing simulated decays of the $Z^0$ boson decaying to a quark and an anti-quark producing two "jets" of particles. The question is, if the jets are from a b-quark (b-jet) or from lighter quarks (l-jet).
3. A "medical" dataset which concers a lifestyle disease in relation to various (transformed) lifestyle variables (reduced in number of variables to match the Aleph b-jet data set).

In the following, we discuss the problem from the b-jet point of view, as this is where the largest size datasets are available. However, we stress that from the point of view of ML, data content (what is being considered) is not essential to know (for now!!!). And knowing the content in details requires domain knowledge, i.e. that you are an expert in the specific field, that the data comes from. This part is very important, but not the focus in this course.

In the end, this exercise is the simple start "outside ML" and moving into the territory of Machine Learning analysis.

### The Data:
The input variables (X) are (used by Aleph for their NN):
* **prob_b**: Probability of being a b-jet from the pointing of the tracks to the vertex.
* **spheri**: Sphericity of the event, i.e. how spherical it is.
* **pt2rel**: The transverse momentum squared of the tracks relative to the jet axis, i.e. width of the jet.
* **multip**: Multiplicity of the jet (in a relative measure).
* **bqvjet**: b-quark vertex of the jet, i.e. the probability of a detached vertex.
* **ptlrel**: Transverse momentum (in GeV) of possible lepton with respect to jet axis (about 0 if no leptons).

Auxilary variables (Z) are (not used by Aleph for their NN):
* energy: Measured energy of the jet in GeV. Should be 45 GeV, but fluctuates.
* cTheta: cos(theta), i.e. the polar angle of the jet with respect to the beam axis. Note, that the detector works best in the central region (|cTheta| small) and less well in the forward regions.
* phi:    The azimuth angle of the jet. As the Aleph detector was essentially uniform in phi, this should not matter (much).

The target variable (Y) is:
* **isb**:    1 if it is from a b-quark and 0, if it is not.

Finally, those before you (the Aleph collaboration in the mid 90'ies) produced a Neural Net (6 input variables, two hidden layers with 10 neurons in each, and 1 output varible) based classification variable, which you can compare to (and compete with?):
* **nnbjet**: Value of original Aleph b-jet tagging algorithm, using only the last six variables (for reference).

In case you choose **the medical data**, the variables to use as input (X) are: **Qsocial, BMI, Roccupat, Rgenetic, Rdietary, and Rhormonn** (reflecting Quantiles and Ratios of medical measurements). The target variable (Y) is (naturally): **TrulyIll**, and you can compare your results to the average of doctors: **DocScore**.


## The Task: Input Feature Ranking (here for LightGBM model):

The following exercise is about Input Feature Ranking. Your task is to give an overview out of the importance of each variable in the model, and their respective contribution to the models perfomance.



* Author: Troels C. Petersen (NBI)
* Email:  petersen@nbi.dk
* Date:   15th of April 2025

In [ ]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
from pandas import Series, DataFrame 
from matplotlib import pyplot as plt
import numpy as np
import seaborn as sns

SavePlots = False

## Import and inspect the data:

In [ ]:
# Read the data and print the variables:
data = pd.DataFrame(np.genfromtxt('../Week1/AlephBtag_MC_train_Nev50000.csv', names=True))
# data = pd.DataFrame(np.genfromtxt('../Week1/Medical_Npatients50000.csv', names=True))

variables = data.columns
print(variables.values)

# Decide on which variables to use for input (X) and what defines the label (Y):
plotting_mask = (variables != 'nnbjet') & (variables != 'energy') & (variables != 'cTheta') & (variables != 'phi')
input_variables = variables[(variables != 'nnbjet') & (variables != 'isb') & (variables != 'energy') & (variables != 'cTheta') & (variables != 'phi')]
input_data      = data[input_variables]
truth_data      = data['isb']
benchmark_data  = data['nnbjet']
print("  Variables used for training: ", input_variables.values)

In [ ]:
pct_truth = truth_data.sum() / truth_data.shape[0]
print(pct_truth)

***

In [ ]:
plotting_data = data[variables[plotting_mask]].sample(frac=1, random_state=42).reset_index(drop=True)

sns.pairplot(plotting_data.head(100), hue='isb')

In [ ]:
corr = input_data.corr()
mask = np.zeros_like(corr)
mask[np.tril_indices_from(mask)] = True
sns.heatmap(corr, cmap='Blues', annot=True, mask=mask.T)

In [ ]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# Dataset is shuffeled before the split (to avoid any ordering). By using a fixed
# random seed number (42), we can rerun and obtain the same result (for reproducibility!).
X_train, X_test, y_train, y_test, bench_train, bench_test = \
    train_test_split(input_data, truth_data, benchmark_data, test_size=0.25, random_state=42,)
    
    
lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_test, y_test, reference=lgb_train)

***

# Classification using LightGBM:


This is a solution example using LightGBM (tree based).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_validate
from lightgbm import LGBMClassifier

def compute_feature_importances(X, y, importance_types=['split', 'gain'], cv_splits=5, top_n=None, plot=True):
    """
    Compute LightGBM feature importances for both 'split' and 'gain' types via cross-validation.

    Parameters:
    -----------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series or np.array
        Target vector
    importance_types : list
        Importance types to compute ('split', 'gain')
    cv_splits : int
        Number of folds for cross-validation
    top_n : int or None
        If set, returns only top-N features ranked by gain
    plot : bool
        If True, generates bar and scatter plots

    Returns:
    --------
    importance_df : pd.DataFrame
        DataFrame with features and their mean importances
    top_features : list
        Top-N features by gain (if top_n is not None)
    """
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
    results = {}

    for imp_type in importance_types:
        model = LGBMClassifier(
            objective='binary',
            metric='mse',
            num_leaves=6,
            verbose=-1,
            importance_type=imp_type
        )

        scores = cross_validate(
            estimator=model,
            X=X,
            y=y,
            cv=kf,
            return_estimator=True,
            scoring=['neg_mean_squared_error'],
            n_jobs=-1
        )

        importances = np.mean(
            [estimator.feature_importances_ for estimator in scores['estimator']],
            axis=0
        )

        feature_names = scores['estimator'][0].feature_name_
        results[imp_type] = pd.Series(importances, index=feature_names)

    # Merge into a single DataFrame
    importance_df = pd.DataFrame({
        'Feature': results['split'].index,
        'Split Importance': results.get('split', pd.Series(np.zeros(len(X.columns)), index=X.columns)),
        'Gain Importance': results.get('gain', pd.Series(np.zeros(len(X.columns)), index=X.columns)),
    }).sort_values(by='Gain Importance', ascending=False).reset_index(drop=True)

    if plot:
        # Bar plot by gain
        plt.figure(figsize=(8, 5))
        sns.barplot(data=importance_df, x='Gain Importance', y='Feature')
        plt.title("Feature Importance by Gain")
        plt.tight_layout()
        plt.show()

        # Scatter plot: Split vs Gain
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=importance_df, x='Split Importance', y='Gain Importance', hue='Feature')
        plt.title("Split vs Gain Feature Importance")
        plt.xscale('log')
        plt.yscale('log')
        plt.tight_layout()
        plt.show()

    top_features = None
    if top_n is not None:
        top_features = importance_df['Feature'].iloc[:top_n].tolist()

    return importance_df, top_features


importance_df_0, top_features_0 = compute_feature_importances(
    X=X_train,
    y=y_train,
    # top_n=10,
    plot=True
)


# N_features = X_train.columns.shape[0]
# importance_df_1, top_features_1 = compute_feature_importances(
#     X=X_train,
#     y=y_train,
#     top_n=(N_features - 3),
#     plot=False
# )

# # Now prune your dataset and retrain
# X_train_pruned = X_train[top_features_1]
# X_test_pruned  = X_test[top_features_1]


In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, roc_curve, auc, confusion_matrix

def prune_retrain_evaluate(
    X_train, X_test, y_train, y_test,
    top_features,
    train_params,
    num_boost_round=1000,
    early_stop_rounds=20
):
    """
    Prune to top_features, retrain a LightGBM model, and evaluate.

    Parameters
    ----------
    X_train, X_test : pd.DataFrame
        Original feature matrices.
    y_train, y_test : array-like
        Targets.
    top_features : list of str
        Columns to keep.
    train_params : dict
        LightGBM train() parameters (incl. 'objective', 'boosting_type', etc.).
    num_boost_round : int
        Maximum number of boosting iterations.
    early_stop_rounds : int
        Rounds of no improvement before early stopping.

    Returns
    -------
    booster : lgb.Booster
        The trained LightGBM model.
    metrics : dict
        Dictionary with keys 'mse', 'roc_auc', 'conf_matrix'.
    """
    # 1) Subset the data
    X_tr = X_train[top_features]
    X_te = X_test[top_features]

    # 2) Create LightGBM datasets
    lgb_tr = lgb.Dataset(X_tr, y_train)
    lgb_ev = lgb.Dataset(X_te, y_test, reference=lgb_tr)

    # 3) Train
    booster = lgb.train(
        train_params,
        lgb_tr,
        num_boost_round=num_boost_round,
        valid_sets=[lgb_ev],
        callbacks=[lgb.early_stopping(early_stop_rounds)]
    )

    # 4) Predict
    y_pred_prob = booster.predict(X_te, num_iteration=booster.best_iteration)
    y_pred      = (y_pred_prob >= 0.5).astype(int)

    # 5) Metrics
    mse       = mean_squared_error(y_test, y_pred_prob)
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    roc_auc   = auc(fpr, tpr)
    conf_mat  = confusion_matrix(y_test, y_pred)

    metrics = {
        'mse': mse,
        'roc_auc': roc_auc,
        'conf_matrix': conf_mat
    }

    return booster, metrics



# Assuming compute_feature_importances and prune_retrain_evaluate are already defined
def feature_pruning_report(X_train, X_test, y_train, y_test, train_params, min_features=1, step=1):
    """
    Iteratively prune features from full set down to min_features,
    retrain the model, and collect metrics.

    Parameters:
    -----------
    X_train, X_test : pd.DataFrame
        Training and test feature matrices.
    y_train, y_test : array-like
        Target vectors.
    train_params : dict
        Parameters for lgb.train.
    min_features : int
        Minimum number of top features to test.
    step : int
        Step size to decrease number of features.

    Returns:
    --------
    report_df : pd.DataFrame
        DataFrame with columns: N_features, MSE, ROC_AUC, TN, FP, FN, TP
    """
    # Full importance once
    importance_df, _ = compute_feature_importances(X_train, y_train, top_n=None, plot=False)
    total_features = importance_df.shape[0]

    records = []

    # Loop over number of features
    for k in range(total_features, min_features - 1, -step):
        top_k = importance_df['Feature'].iloc[:k].tolist()

        # Prune, retrain, and evaluate
        booster, metrics = prune_retrain_evaluate(
            X_train, X_test, y_train, y_test,
            top_features=top_k,
            train_params=train_params
        )

        # Unpack confusion matrix
        tn, fp, fn, tp = metrics['conf_matrix'].ravel()

        records.append({
            'N_features': k,
            'MSE': metrics['mse'],
            'ROC_AUC': metrics['roc_auc'],
            'TN': tn,
            'FP': fp,
            'FN': fn,
            'TP': tp
        })

    report_df = pd.DataFrame(records)
    return report_df

# Example usage:
train_params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'num_leaves': 8,
    'max_depth': 12,
    # 'min_data_in_leaf': 10,
    'learning_rate': 0.1
}

report_df = feature_pruning_report(
    X_train, X_test, y_train, y_test,
    train_params=train_params,
    min_features=3,
    step=1, 
)

# Display the report 
display(report_df)


Best overall (6 features):
- Lowest MSE (0.07206)
- Highest ROC-AUC (0.9353) -> Keeping all six top features gives best predictive performance.


Confusion-matrix shifts:
- As you prune, false negatives (FN) increase more rapidly than false positives (FP) decrease. In practical terms, using fewer features makes the model miss more true positives.
- At 6 features: FN = 810 → at 3 features: FN = 902 (↑92 missed positives).

## Hyperparamter Optimisation

In [ ]:
# X_train_pruned = X_train[top_features_1]
# X_test_pruned  = X_test[top_features_1]
X_train_pruned = X_train.copy()
X_test_pruned  = X_test.copy()

### GridSearchCV

In [ ]:
# from sklearn.model_selection import GridSearchCV
# from lightgbm import LGBMClassifier

# param_grid = {
#     'num_leaves': [16, 32, 64],
#     'max_depth': [4, 6, 8],
#     'feature_fraction': [0.7, 0.9, 1.0],
#     'bagging_fraction': [0.7, 0.9, 1.0],
#     'learning_rate': [0.01, 0.05, 0.1],
# }

# grid = GridSearchCV(
#     LGBMClassifier(
#         objective='binary',
#         boosting_type='gbdt',
#         verbose=-1,
#     ),
#     param_grid,
#     cv=5,
#     scoring='roc_auc',
#     n_jobs=-1,
# )

# grid.fit(X_train_pruned, y_train)
# print("Best hyper­parameters:", grid.best_params_)
# print("Best CV ROC-AUC:", grid.best_score_)


In [ ]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

def objective(trial):
    params = {
        # 'objective': 'binary',
        'boosting_type': 'gbdt',
        'verbose': -1,
        # 'n_jobs': -1,
        'num_leaves': trial.suggest_int('num_leaves', 3, 24),
        'max_depth': trial.suggest_int('max_depth', 3, 18),
        # 'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        # 'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        # 'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        # 'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
    }

    model = LGBMClassifier(**params)

    # Use cross-validation with AUC
    auc_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    return auc_scores.mean()


study = optuna.create_study(direction='maximize', study_name='lgbm_bayes_opt')
study.optimize(objective, n_trials=50,
            #    timeout=600,
               )  # 50 trials or 10 min



print("Best AUC:", study.best_value)
print("Best hyperparameters:", study.best_params)

best_model = LGBMClassifier(**study.best_params)
best_model.fit(X_train, y_train)

# # Evaluate on the test set
# y_pred = best_model.predict(X_test)
# roc_auc = roc_auc_score(y_test, y_pred)
# print("Test ROC-AUC:", roc_auc)

In [ ]:
from os import makedirs
model_dir = './_model_cache/'
makedirs(model_dir, exist_ok=True)
best_model.booster_.save_model(model_dir + 'model_lgbm_bayes_opt');

In [ ]:
rel_dif = np.abs((sum(best_model.predict(X_test)) - truth_data.sum()) / truth_data.sum())
print(rel_dif)

In [ ]:
best_model.predict(X_test)
y_score = best_model.predict(X_test)
y_pred  = [1 if pred > 0.1 else 0 for pred in y_score]       

fpr, tpr, _ = roc_curve(y_test, y_score)                  # False/True Positive Rate for our model
fpr_nnbjet, tpr_nnbjet, _ = roc_curve(y_test, bench_test)  # False/True Positive Rate for Aleph NNbjet

auc_score = auc(fpr,tpr)                        # This is the AUC score for our model
auc_score_nnbjet = auc(fpr_nnbjet, tpr_nnbjet)  # This is the AUC score for Aleph NNbjet


fig = plt.figure(figsize = [10,10])
plt.title('Model Comparison (ROC curves)', size = 16)
plt.plot(fpr, tpr, label=f'Our LightGBM model (AUC = {auc_score:5.3f})')
plt.plot(fpr_nnbjet, tpr_nnbjet, label = f'Aleph NNbjet (AUC = {auc_score_nnbjet:5.3f})')
plt.legend(fontsize=16)
plt.xlabel('False Postive Rate', size=16)
plt.ylabel('True Positive Rate', size=16)
plt.show()


# roc_auc = roc_auc_score(y_test, y_pred)
# print("Test ROC-AUC:", roc_auc)

# # Evaluate on the test set
# # y_pred = best_model.predict(X_train_pruned)

# # roc_auc = roc_auc_score(y_test, y_pred)
# # print("Test ROC-AUC:", roc_auc)


In [ ]:
optuna.visualization.plot_optimization_history(study).show()
optuna.visualization.plot_param_importances(study).show()

### SHAP

In [ ]:
import shap

# 1. Fit your final model (e.g. with tuned hyper­params)
model = LGBMClassifier(**grid.best_params_).fit(X_train_pruned, y_train)

# 2. Create a SHAP explainer
explainer = shap.Explainer(model, X_train_pruned)

# 3. Compute SHAP values
shap_values = explainer(X_test_pruned)

# 4. Summary plot (global importance & directionality)
shap.summary_plot(shap_values, X_test_pruned)

# 5. Dependence plot for feature 'f_i'
shap.dependence_plot('f_i', shap_values, X_test_pruned)


## Training

In [ ]:
# # train_params = {
#     # 'objective': 'boosting',
# #     'boosting_type': 'gbdt', # Traditional Gradient Boosting tree, we are combining many 'weak' learners here!
# #     'objective': 'binary',   # The outcome is binary, b-quark or not
# #     'num_leaves': 6,         # Set a maximum tree leaves to avoid overfitting
# #     'verbose': 1,            # Level of output. Can be set to -1 to suppress the output
# # }

# # Train the model:
# model = lgb.train(train_params,
#                 lgb_train,                         
#                 num_boost_round=10_000,              
#                 valid_sets=lgb_eval,               
#                 callbacks=[early_stopping(250)])    

# # Make predictions.
# # NOTE the difference between 'score' (continuous in ]0,1[) and 'predictions' (integer: 0 or 1):
# # Also NOTE that you can choose where to set the threshold (here set to 0.1)
# y_score = model.predict(X_test, num_iteration=model.best_iteration)  # Scores are floats in the range ]0,1[.
# y_pred  = [1 if pred > 0.1 else 0 for pred in y_score]               # Classify b-quark yes or no (for comparison). 

In [ ]:
# # Evaluate:
# fpr, tpr, _ = roc_curve(y_test, y_score)                  # False/True Positive Rate for our model
# fpr_nnbjet, tpr_nnbjet, _ = roc_curve(y_test, bench_test)  # False/True Positive Rate for Aleph NNbjet

# # We can now calculate the Area-Under-the-Curve (AUC) scores of these ROC-curves:
# auc_score = auc(fpr,tpr)                        # This is the AUC score for our model
# auc_score_nnbjet = auc(fpr_nnbjet, tpr_nnbjet)  # This is the AUC score for Aleph NNbjet

# # Let's plot the ROC curves for these results:
# fig = plt.figure(figsize = [10,10])
# plt.title('Model Comparison (ROC curves)', size = 16)
# plt.plot(fpr, tpr, label=f'Our LightGBM model (AUC = {auc_score:5.3f})')
# plt.plot(fpr_nnbjet, tpr_nnbjet, label = f'Aleph NNbjet (AUC = {auc_score_nnbjet:5.3f})')
# plt.legend(fontsize=16)
# plt.xlabel('False Postive Rate', size=16)
# plt.ylabel('True Positive Rate', size=16)
# plt.show()

## Questions:

1. Try to determine an input feature ranking first by using the **build-in function** for this in LightGBM. Yes, look at the documentation and find the one line of code that does this. Can you also find - and understand - the description of how it reaches these results?

2. Now try to get an input feature ranking using **permutation invariance**. Think of an ML package that contains a lot of different ML tools, and check if this package does not have a tool for doing so. Do you get a similar ranking as for the above method?

3. Finally, calculate SHAP values and use these (using Sum_i |SHAP_i |) to determine a ranking of the input features. Again, compare it to the two above. Do they agree?

4. Try to rerun the model twice with only 5 input parameters: Missing the highest and lowest ranking feature. Do you see a significant difference in the performance between these two cases? And how about when comparing to the 6 input feature case?


## Learning points:

1. You should first of all understand the concept of “feature ranking”.

2. You should also be capable of determining the feature ranking for a given algorithm in multiple ways, specifically using permutation invariance (PI) and SHAP values.

3. Finally, you should understand that while PI and other build-in methods gives an average input feature ranking, the SHAP values are capable of giving an individual input feature ranking for each case.


In [ ]:
# Plot feature importance 

lgb.plot_importance(booster=model,
                    # importance_type="gain", 
                    figsize=(10,6),
                    # title="LightGBM Feature Importance (Gain)",
                    )
plt.show()